In [14]:
# %matplotlib inline
from utils.opconNosepokeFunctions import *


toneStartMarker = 23
trialStartMarker = 81
trialHitMarker = 51
trialMissMarker = 86
trialUnrewMarker = 87
sessionStartMarker = 61
rewardProbMarker = 83
trialProbMarker = 88
lickMarker = 22
list_sessionMarker = [12, 13]

# load files

In [33]:
animal = 'Bidoof'
box = 2
date = '2025-07-01'
# end_date = '2026-05-19'
# path = f"L:/4portProb/Box{box}/{animal}/"       # for any les animal
path = f"L:/4portProb_ephys/Box{box}_ephys/{animal}/Behavior/"      # for ephys only

files = exclude_files(get_files(path))
set_cwd(path)
files = [file for file in files if (file >=f'{animal}-*.dat')] #& (file <= f'{animal}-{end_date}*.dat')]

# files = [r"C:\Users\dlab\Documents\OpCon v2.2 Nosepoke\Bhindi-2025-02-27-15-38-26.dat"]
# set_cwd(r'C:\Users\dlab\Documents\OpCon v2.2 Nosepoke')
files.sort()
print(files)
list_sessionMarker = [13]

# df = merge_files_to_df(files)

# sessdf = trializer_v3(df, list_sessionMarker,
#                       trialStartMarker, trialHitMarker,
#                       trialMissMarker, sessionStartMarker,
#                       trialProbMarker, rewardProbMarker,
#                       lickMarker, animal)
from utils.trializer_v4_synced import trializer_v4_synced
sessdf = trializer_v4_synced(files, animal)
sessdf['datetime'] = pd.to_datetime(sessdf.eptime, unit='s')
SESSION_GAP_SEC = 40 * 60

# if things look wonky maybe the data is not sorted!
# (it should be though the dat files are sorted per animal)
# sessdf = sessdf.sort_values(["animal", "eptime"]).copy()
gap_sec = sessdf.groupby("animal")["eptime"].diff()
new_session = gap_sec.isna() | (gap_sec > SESSION_GAP_SEC)

# Session numbers per animal: 1, 2, 3, ...
sessdf["session"] = new_session.groupby(sessdf["animal"]).cumsum().astype(int)
sessdf.rename(columns={'trial_rewprob':'rewprob'}, inplace = True)
# if you want to add block numbers, uncomment this section
sessdf['sess_block'] = 0
thing = sessdf.groupby(['animal', 'task', 'session']).datetime.head(1).astype('datetime64[ns]').diff().dt.seconds>2400
sessdf.loc[thing[thing == True].index, 'sess_block'] = 1
sessdf['sess_block'] = sessdf.groupby(['animal', 'task']).sess_block.cumsum()

['Bidoof-2026-02-24-10-24-27.dat', 'Bidoof-2026-02-25-10-24-27.dat', 'Bidoof-2026-02-26-10-24-27.dat', 'Bidoof-2026-02-27-10-24-27.dat', 'Bidoof-2026-02-28-10-24-27.dat', 'Bidoof-2026-03-01-10-24-27.dat', 'Bidoof-2026-03-02-10-24-27.dat', 'Bidoof-2026-03-03-10-24-27.dat', 'Bidoof-2026-03-04-10-24-27.dat', 'Bidoof-2026-03-05-10-24-27.dat', 'Bidoof-2026-03-06-10-24-27.dat', 'Bidoof-2026-03-07-10-24-27.dat', 'Bidoof-2026-03-08-10-24-27.dat', 'Bidoof-2026-03-09-10-24-27.dat', 'Bidoof-2026-03-10-10-24-27.dat', 'Bidoof-2026-03-10-16-42-57.dat', 'Bidoof-2026-03-11-16-42-57.dat', 'Bidoof-2026-03-11-20-02-48.dat', 'Bidoof-2026-03-12-20-02-48.dat', 'Bidoof-2026-03-13-20-02-48.dat', 'Bidoof-2026-03-14-20-02-48.dat', 'Bidoof-2026-03-15-20-02-48.dat', 'Bidoof-2026-03-16-20-02-48.dat', 'Bidoof-2026-03-17-20-02-48.dat', 'Bidoof-2026-03-18-20-02-48.dat', 'Bidoof-2026-03-19-20-02-48.dat', 'Bidoof-2026-03-20-20-02-48.dat', 'Bidoof-2026-03-21-20-02-48.dat', 'Bidoof-2026-03-22-08-24-15.dat', 'Bidoof-2026-

In [16]:
sessdf['week_task'] = sessdf.groupby(['animal', 'task'])['datetime'].transform(
    lambda x: ((x - x.min()).dt.days // 7) + 1
)
sessdf['day_task'] = sessdf.groupby('animal')['datetime'].transform(
    lambda x: ((x - x.min()).dt.days) + 1
)

# Nosepoke session analysis

In [133]:
# %matplotlib nbagg
# this plots blockwise raster of choice
%matplotlib qt

import ipywidgets as widgets
from IPython.display import display
from matplotlib.patches import Rectangle
from utils.plotSettings import *
# import matplotlib.pyplot as plt
import seaborn as sns

# Create a function to update the plot based on user input
def update_plot(session_range):
    sessdf_filtered = sessdf[sessdf['block_num'].between(*session_range)]
    markers = {0: "$o$", 1: "o"}   # for reward-wise raster, useful for initial stages or zooming into data
#     markers = {0: 2, 1:3}
#     fig = plt.figure(figsize=(15, 8))
    g = sns.relplot(data=sessdf_filtered, y='port', x='trial', 
                    hue='rewprob',
                    aspect=11.7/5.5,
                    linewidth=0.75, s=100, palette = 'Blues', marker = '|', # change '|' to markers for reward-wise raster
                    # markers=markers, style='reward',
                    hue_norm=(-20, 80))
    axes = g.axes.flatten()
    last = sessdf.groupby('block_num')['trial'].max().to_list()
    start = sessdf.groupby('block_num')['trial'].min().to_list()
    highport = [ind+1 for i in sessdf.groupby('block_num').head(1).rewprobfull for ind, j in enumerate(i.strip('[]').split()) if j=='80']
    for ax in axes:
        for ind, l in enumerate(last):
            ax.axvline(l, linewidth=0.25, color='grey')
#             ax.axvline(, linewidth = 1, color = 'red')
            ax.set_yticks([1, 2, 3, 4])
            ax.add_patch(Rectangle((start[ind]-1, highport[ind]-0.5), l-start[ind], 1, fc = 'xkcd:blue', alpha = 0.1))
    
    plt.ylim(0.5, 4.5)
    plt.title('Choices', y=1.05)
    g._legend.set_title('Rew. %')
    # g._legend.texts[0].set_text("Reward %")
    # g._legend.texts[5].set_text("Outcome")
    plt.xticks(ticks = [], labels = [])

    plt.xlabel(' ', labelpad=20)
    plt.ylabel('Port')
    
    # Calculate x-axis limits based on the displayed sessions
    x_limit = (sessdf_filtered['trial'].min(), sessdf_filtered['trial'].max())
    plt.xlim(*x_limit)
    sns.despine(bottom = True)
#     plt.savefig('C:/Users/dlab/OneDrive - Indian Institute of Science/Drawings/plots/15012024/eg.svg', dpi = 600)
    plt.show()

# Define the session range using a widget
session_range_slider = widgets.IntRangeSlider(
    value=[sessdf['block_num'].min(), sessdf['block_num'].max()],
    min=sessdf['block_num'].min(),
    max=sessdf['block_num'].max(),
    step=1,
    description='Block Range:',
    continuous_update=False
)

# Create an interactive plot using interactive
interactive_plot = widgets.interactive(update_plot, session_range=session_range_slider)

# Display the interactive plot
display(interactive_plot)
plt.xticks(ticks = [], labels = [])
# plt.tight_layout()
# sns.set_context('talk')
# plt.title('Blissey - DLS then DMS lesion')

interactive(children=(IntRangeSlider(value=(0, 1492), continuous_update=False, description='Block Range:', max…

([], [])

In [ ]:
# this plots blockwise raster of choice
import ipywidgets as widgets
from IPython.display import display
from matplotlib.patches import Rectangle
from matplotlib import transforms
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib qt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import numpy as np

band_cmap = 'Purples'       # any sequential matplotlib cmap, e.g. 'Oranges'
band_range = (0.1, 0.6) # part of the cmap to use: lightest (lowest prob) -> darkest (highest prob)
prob_range = None   # reward % mapped to the ends of band_range; set to None to use the min/max prob in the data
rect_alpha = 0.6          # opaque bands keep the shades distinct; lower range above instead of alpha to lighten
r_tick_color = 'xkcd:black'       # rewarded: tall black tick
nr_tick_color = 'xkcd:black'   # unrewarded: short grey tick
rew_lw, unrew_lw = 1.25, 0.8 # lw
rew_h, unrew_h = 0.2, 0.1   # half-heights of the ticks
scalebar_frac = 0.1   # scale bar is the largest 1/2/5 x 10^k trials that fits in this fraction of the visible x-range

# one shade per reward probability, placed along the cmap in proportion to its value
rewprob_vals = sorted(sessdf['rewprob'].dropna().unique())
pmin, pmax = prob_range if prob_range is not None else (min(rewprob_vals), max(rewprob_vals))
frac = (np.array(rewprob_vals) - pmin) / ((pmax - pmin) or 1)
shades = plt.get_cmap(band_cmap)(band_range[0] + frac * (band_range[1] - band_range[0]))
rew_palette = {p: c for p, c in zip(rewprob_vals, shades)}

def nice_length(span, frac=scalebar_frac):
    # largest 1, 2 or 5 x 10^k that is <= frac * span
    target = max(span * frac, 1)
    mag = 10 ** np.floor(np.log10(target))
    for m in (5, 2, 1):
        if m * mag <= target:
            return m * mag
    return mag

# Create a function to update the plot based on user input
def update_plot(session_range):
    sessdf_filtered = sessdf[sessdf['block_num'].between(*session_range)].copy()
    fig, ax = plt.subplots(figsize=(14, 3))
    # tick height encodes outcome, all centred on the port
    rew = sessdf_filtered[sessdf_filtered['reward'] == 1]
    unrew = sessdf_filtered[sessdf_filtered['reward'] == 0]
    ax.vlines(rew['trial'], rew['port']-rew_h, rew['port']+rew_h, color=r_tick_color, lw=rew_lw, zorder=3)
    ax.vlines(unrew['trial'], unrew['port']-unrew_h, unrew['port']+unrew_h, color=nr_tick_color, lw=unrew_lw, zorder=3)

    last = sessdf.groupby('block_num')['trial'].max().to_list()
    start = sessdf.groupby('block_num')['trial'].min().to_list()
    # per-block list of reward probs for ports 1-4
    block_probs = [[float(j) for j in i.strip('[]').split()] for i in sessdf.groupby('block_num').head(1).rewprobfull]
    for ind, l in enumerate(last):
        # background bands, slightly shorter than 1 so each port row has a white gap
        for port, p in enumerate(block_probs[ind], start=1):
            if p in rew_palette:
                ax.add_patch(Rectangle((start[ind]-0.5, port-0.4), l-start[ind]+1, 0.8,
                                       fc=rew_palette[p], alpha=rect_alpha, ec='none', zorder=0))
        ax.axvline(l+0.5, color='white', lw=1, zorder=1)

    ax.set_ylim(0.5, 4.5)
    ax.invert_yaxis()   # port 1 on top
    ax.set_yticks([1, 2, 3, 4], ['Port 1', 'Port 2', 'Port 3', 'Port 4'])
    ax.tick_params(left=False)
    ax.set_xticks([])
    sns.despine(ax=ax, left=True, bottom=True)

    # scale bar instead of an x-axis: x in data units, y in axes units, so it stays
    # below the plot and is redrawn to a sensible length whenever you zoom or pan
    trans = transforms.blended_transform_factory(ax.transData, ax.transAxes)
    bar, = ax.plot([], [], color='k', lw=1.5, transform=trans, clip_on=False)
    bar_label = ax.text(0, -0.1, '', transform=trans, ha='center', va='top', fontsize='small')

    def update_scalebar(ax):
        x0, x1 = ax.get_xlim()
        n = nice_length(x1 - x0)
        xs = x0 + 0.01 * (x1 - x0)
        bar.set_data([xs, xs + n], [-0.05, -0.05])
        bar_label.set_position((xs + n / 2, -0.1))
        bar_label.set_text(f'{n:g} trial' + ('' if n == 1 else 's'))

    ax.callbacks.connect('xlim_changed', update_scalebar)
    x_limit = (sessdf_filtered['trial'].min() - 0.5, sessdf_filtered['trial'].max() + 0.5)
    ax.set_xlim(*x_limit)   # triggers update_scalebar

    # single legend row above the plot: bands match the background, lines match the ticks
    handles = [Patch(fc=c, alpha=rect_alpha, ec='none', label=f'{p:g}%') for p, c in rew_palette.items()]
    outcome_handles = [Line2D([], [], color=r_tick_color, lw=rew_lw*1., label='Rewarded'),
                       Line2D([], [], color=nr_tick_color, lw=unrew_lw*1., label='Unrewarded')]
    ax.legend(handles=handles + outcome_handles, ncol=len(handles) + 2, loc='lower center',
              bbox_to_anchor=(0.5, 1.02), frameon=False, handlelength=1.2, columnspacing=1)
    # ax.legend(handles=handles, ncol=len(handles), loc='lower center',
    #               bbox_to_anchor=(0.5, 1.02), frameon=False, handlelength=1.2, columnspacing=1)
    plt.tight_layout()
    plt.show()

# Define the session range using a widget
session_range_slider = widgets.IntRangeSlider(
    value=[251, 253],
    min=sessdf['block_num'].min(),
    max=sessdf['block_num'].max(),
    step=1,
    description='Block Range:',
    continuous_update=False
)

# Create an interactive plot using interactive
interactive_plot = widgets.interactive(update_plot, session_range=session_range_slider)

# Display the interactive plot
display(interactive_plot)

interactive(children=(IntRangeSlider(value=(251, 253), continuous_update=False, description='Block Range:', ma…

In [38]:
# same choice raster as above, plus a lower axis with each block's reward probs as bars (ports 1-4 left to right)
# run the cell above first: reuses its settings, rew_palette and nice_length
bar_width = 10   # width of each port's bar, in trials (same for every block)
bar_gap = 2     # gap between neighbouring port bars, in trials; the group of 4 is centred on its block

def update_plot_bars(session_range):
    sessdf_filtered = sessdf[sessdf['block_num'].between(*session_range)].copy()
    fig, (ax, bax) = plt.subplots(2, 1, figsize=(14, 4.5), sharex=True, height_ratios=[3, 1.2])
    # tick height encodes outcome, all centred on the port
    rew = sessdf_filtered[sessdf_filtered['reward'] == 1]
    unrew = sessdf_filtered[sessdf_filtered['reward'] == 0]
    ax.vlines(rew['trial'], rew['port']-rew_h, rew['port']+rew_h, color=r_tick_color, lw=rew_lw, zorder=3)
    ax.vlines(unrew['trial'], unrew['port']-unrew_h, unrew['port']+unrew_h, color=nr_tick_color, lw=unrew_lw, zorder=3)

    last = sessdf.groupby('block_num')['trial'].max().to_list()
    start = sessdf.groupby('block_num')['trial'].min().to_list()
    # per-block list of reward probs for ports 1-4
    block_probs = [[float(j) for j in i.strip('[]').split()] for i in sessdf.groupby('block_num').head(1).rewprobfull]
    bar_x, bar_h, bar_c = [], [], []
    for ind, l in enumerate(last):
        # background bands, slightly shorter than 1 so each port row has a white gap
        for port, p in enumerate(block_probs[ind], start=1):
            if p in rew_palette:
                ax.add_patch(Rectangle((start[ind]-0.5, port-0.4), l-start[ind]+1, 0.8,
                                       fc=rew_palette[p], alpha=rect_alpha, ec='none', zorder=0))
        # lower axis: fixed-width bars, group centred on the block, bar height = reward %
        centre = (start[ind] + l) / 2
        n_ports = len(block_probs[ind])
        for port, p in enumerate(block_probs[ind], start=1):
            bar_x.append(centre + (port - (n_ports + 1) / 2) * (bar_width + bar_gap))
            bar_h.append(p)
            bar_c.append(rew_palette.get(p, 'lightgrey'))
        ax.axvline(l+0.5, color='white', lw=1, zorder=1)
        bax.axvline(l+0.5, color='lightgrey', lw=0.5, zorder=0)
    # same alpha as the raster bands so the shades match exactly
    bax.bar(bar_x, bar_h, width=bar_width, color=bar_c, alpha=rect_alpha, ec='none', zorder=2)

    ax.set_ylim(0.5, 4.5)
    ax.invert_yaxis()   # port 1 on top
    ax.set_yticks([1, 2, 3, 4], ['Port 1', 'Port 2', 'Port 3', 'Port 4'])
    ax.tick_params(left=False, bottom=False)
    sns.despine(ax=ax, left=True, bottom=True)

    bax.set_ylim(0, 80)
    bax.set_yticks([0, 40, 80])
    bax.set_ylabel('Rew. %\n(ports 1-4)', fontsize='small')
    bax.set_xticks([])
    sns.despine(ax=bax, bottom=True)

    # scale bar under the raster, redrawn on zoom/pan
    trans = transforms.blended_transform_factory(ax.transData, ax.transAxes)
    bar, = ax.plot([], [], color='k', lw=1.5, transform=trans, clip_on=False)
    bar_label = ax.text(0, -0.08, '', transform=trans, ha='center', va='top', fontsize='small')

    def update_scalebar(a):
        x0, x1 = a.get_xlim()
        n = nice_length(x1 - x0)
        xs = x0 + 0.01 * (x1 - x0)
        bar.set_data([xs, xs + n], [-0.04, -0.04])
        bar_label.set_position((xs + n / 2, -0.08))
        bar_label.set_text(f'{n:g} trial' + ('' if n == 1 else 's'))

    ax.callbacks.connect('xlim_changed', update_scalebar)
    x_limit = (sessdf_filtered['trial'].min() - 0.5, sessdf_filtered['trial'].max() + 0.5)
    ax.set_xlim(*x_limit)   # shared x, triggers update_scalebar

    # handles = [Patch(fc=c, alpha=rect_alpha, ec='none', label=f'{p:g}%') for p, c in rew_palette.items()]
    # ax.legend(handles=handles, ncol=len(handles), loc='lower center',
    #           bbox_to_anchor=(0.5, 1.02), frameon=False, handlelength=1.2, columnspacing=1)
    handles = [Patch(fc=c, alpha=rect_alpha, ec='none', label=f'{p:g}%') for p, c in rew_palette.items()]
    outcome_handles = [Line2D([], [], color=r_tick_color, lw=rew_lw*1., label='Rewarded'),
                        Line2D([], [], color=nr_tick_color, lw=unrew_lw*1., label='Unrewarded')]
    ax.legend(handles=handles + outcome_handles, ncol=len(handles) + 2, loc='lower center',
                bbox_to_anchor=(0.5, 1.02), frameon=False, handlelength=1.2, columnspacing=1)
    plt.tight_layout()
    fig.subplots_adjust(hspace=0.35)   # room for the scale bar between the two axes
    plt.show()

session_range_slider_bars = widgets.IntRangeSlider(
    value=[251, 253],
    min=sessdf['block_num'].min(),
    max=sessdf['block_num'].max(),
    step=1,
    description='Block Range:',
    continuous_update=False
)
display(widgets.interactive(update_plot_bars, session_range=session_range_slider_bars))

interactive(children=(IntRangeSlider(value=(251, 253), continuous_update=False, description='Block Range:', ma…

In [6]:
sessdf[mask].groupby('animal').count()

,task,block_num,trial,port,reward,trial_rewprob,rewprobfull,poke_in_time,poke_out_time,first_lick_time,last_lick_time,eptime,datetime
animal,,,,,,,,,,,,,
Sableye,62,62,62,62,26,62,62,62,62,0,0,62,62
Torchic,472,472,472,472,228,228,472,36,472,252,252,472,472


In [5]:
# check if there are any duplicated trials - this means there's something wrong with the box
mask = sessdf.duplicated(subset = ['animal', 'poke_in_time', 'eptime'], keep = False)
sessdf[mask]

,animal,task,block_num,trial,port,reward,trial_rewprob,rewprobfull,poke_in_time,poke_out_time,first_lick_time,last_lick_time,eptime,datetime
0,Torchic,unstr,0,0,1,NaN,NaN,[46 80 15 46],NaN,311960.0,NaN,NaN,1755299953,2025-08-15 23:19:13
1,Torchic,unstr,0,1,3,NaN,NaN,[46 80 15 46],NaN,14485.0,NaN,NaN,1755299953,2025-08-15 23:19:13
2,Torchic,unstr,0,2,4,1.0,80.0,[46 80 15 46],NaN,85819.0,1893.0,2947.0,1755299953,2025-08-15 23:19:13
3,Torchic,unstr,0,3,1,NaN,NaN,[46 80 15 46],NaN,311960.0,NaN,NaN,1755299955,2025-08-15 23:19:15
4,Torchic,unstr,0,4,3,NaN,NaN,[46 80 15 46],NaN,14485.0,NaN,NaN,1755299955,2025-08-15 23:19:15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
739375,Sableye,dms,950,156935,2,1.0,46.0,[80 46 46 15],704209.0,704210.0,NaN,NaN,1764444271,2025-11-29 19:24:31
739376,Sableye,dms,950,156936,2,NaN,46.0,[80 46 46 15],671595.0,671596.0,NaN,NaN,1764444239,2025-11-29 19:23:59
739378,Sableye,dms,950,156938,2,0.0,46.0,[80 46 46 15],914724.0,914725.0,NaN,NaN,1764444482,2025-11-29 19:28:02
739379,Sableye,dms,950,156939,2,1.0,46.0,[80 46 46 15],704209.0,704210.0,NaN,NaN,1764444271,2025-11-29 19:24:31


In [230]:
#plot number of times each port was chosen/total trials in block as a function of rewprob at that port, for all trials indf; average across blocks
port_choices = [1, 2, 3, 4]
#group by unique rewprob values, find unique rewprob values across all ports first
colors_ports = ['red', 'green', 'blue', 'purple']
unique_rewprobs = np.unique(sessdf['rewprob'])
# print(unique_rewprobs)
#remove nan from unique_rewprobs
unique_rewprobs = unique_rewprobs[~np.isnan(unique_rewprobs)]
print(unique_rewprobs)
#remove rows with nan rewprob
sessdf = sessdf[~sessdf['rewprob'].isna()]

#for every block, calculate number of times each port was chosen/total trials in block for each unique rewprob value
nblocks = sessdf['session'].nunique()
port_choice_matrix = np.zeros((len(unique_rewprobs), len(port_choices), nblocks))
#for all blocks
#save all rewprobfull per block in a list
for block in range(1, sessdf['session'].nunique()+1):
    block_data = sessdf[sessdf['session'] == block]
    #for all ports
    for p, port in enumerate(port_choices):
        port_data = block_data[block_data['port'] == port]
        #for all unique rewprobs
        for r, rewprob in enumerate(unique_rewprobs):
            n_choices = len(port_data[port_data['rewprob'] == rewprob])
            port_choice_matrix[r, p, block-1] = n_choices

bias_matrix = np.zeros((port_choice_matrix.shape[0], port_choice_matrix.shape[1]))
counter = np.zeros((port_choice_matrix.shape[0], port_choice_matrix.shape[1]))

for p in range(port_choice_matrix.shape[1]):
    for r in range(port_choice_matrix.shape[0]):
        bias_matrix[r, p] = 0
        for b in range(port_choice_matrix.shape[2]):
            ntrls = np.sum(port_choice_matrix[:,:,b])
            if ntrls != 0:
                bias_matrix[r, p] += port_choice_matrix[r, p, b] / ntrls
            if port_choice_matrix[r, p, b] > 0:
                counter[r, p] += 1

#divide fin_mat by counter
for p in range(port_choice_matrix.shape[1]):
    for r in range(port_choice_matrix.shape[0]):
        if counter[r, p] > 0:
            bias_matrix[r, p] = bias_matrix[r, p] / counter[r, p]
print('Bias matrix:', bias_matrix)


#plot as line plots for each port
plt.figure(figsize=(7, 6))
for p, port in enumerate(port_choices):
    plt.plot([0,1,2,3], bias_matrix[:, p], label=f'Port {port}', marker='o', linewidth=2, color=colors_ports[p])
plt.xlabel('Reward Probability')
plt.xticks([0,1,2,3], labels=[int(r) for r in unique_rewprobs])
plt.ylabel(f'Fraction chosen (averaged across {nblocks} blocks)')
plt.title(f'{animal}')
plt.legend()
plt.show()

n_rows, n_cols = bias_matrix.shape
rpf = [10,15,46,80]
labels = [i+1 for i in range(n_cols)]

# Angles for radar
angles = np.linspace(0, 2*np.pi, n_cols, endpoint=False)
angles = np.concatenate([angles, [angles[0]]])  # close loop

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
for i in range(n_rows):
    values = bias_matrix[i]
    values = np.concatenate([values, [values[0]]])  # close loop

    ax.plot(angles, values, linewidth=2, label=f'prob {rpf[i]}', color=colors_ports[i])
    # ax.fill(angles, values, alpha=0.15)

ax.set_thetagrids(np.degrees(angles[:-1]), labels)
#reduce the yticks to 3
# ax.set_yticks([])
ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1.1))
plt.tight_layout()
plt.show()
 

[10. 15. 46. 80.]
Bias matrix: [[0.09170014 0.04574203 0.08286635 0.10012489]
 [0.11473001 0.08672058 0.06668227 0.00601408]
 [0.27532335 0.19790004 0.14718805 0.11431743]
 [0.25746868 0.21304803 0.16927896 0.14373582]]


In [ ]:
# add columns to make transition matrix
sessdf['choice_t1'] = sessdf.groupby(['animal','session']).port.shift(-1)
sessdf['choice_t2'] = sessdf.groupby(['animal','session']).port.shift(-2)
sessdf['shift_t0'] = (sessdf['choice_t1']==sessdf['port']).replace({True: 0, False: 1})
sessdf['shift_t1'] = (sessdf['choice_t2']==sessdf['port']).replace({True: 0, False: 1})

C:\Users\dlab\AppData\Local\Temp\ipykernel_16272\2834681967.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sessdf['shift_t0'] = (sessdf['choice_t1']==sessdf['port']).replace({True: 0, False: 1})
C:\Users\dlab\AppData\Local\Temp\ipykernel_16272\2834681967.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sessdf['shift_t1'] = (sessdf['choice_t2']==sessdf['port']).replace({True: 0, False: 1})


In [ ]:
import matplotlib.pyplot as plt
plt.figure()
set_cwd('C:/Users/dlab/rishika_sim')
from utils.plotSettings import *
parula = get_parula_cmap()

ax = plt.subplot(111)
ratdf = sessdf[ ~(sessdf.duplicated(subset = ['animal', 'session', 'trialstart', 'trialend', 'eptime'], keep = False))]
# raltzdf = raltzdf.groupby('session').filter(lambda x: x.reward.size >= trialsinsess).groupby('session').head(trialsinsess)
sns.heatmap(pd.crosstab(ratdf.port, ratdf.choice_t1, normalize = 'index'), 
            ax = ax,
            cmap = parula,
            annot = True,
            fmt = '.2f',
            vmin = 0.0,
            vmax = 0.1,
            square = True,
            # mask = np.eye(4),
            xticklabels = np.arange(1,5), yticklabels = np.arange(1,5))
# ax.patch.set_facecolor('white')
ax.set_xlabel('Choice at trial t+1', fontsize = 'x-large')
ax.set_ylabel('Choice at trial t', fontsize = 'x-large')
ax.tick_params(axis = u'both', which = u'both', length = 0, labelsize = 'large', color = 'grey')

In [ ]:
# plot reward rate / regret
window = 5
sessdf['rr'] = sessdf.groupby('session', as_index = False).reward.rolling(window, center=True).mean().reward
sessdf['regret'] = abs(sessdf['rewprob']-80)
ax= plt.subplot(111)
rr_mean_dict = {}
trialsinsess = 100
filtered = (sessdf.groupby('session')).filter(lambda x: x.reward.size >= trialsinsess)

# take only those sessions
filtered = filtered.groupby('session').head(trialsinsess)


# convert to numpy array
g = filtered.groupby('session').cumcount()
L = np.array(filtered.set_index(['session',g])
    .unstack(fill_value=0)
    .stack(future_stack = True).groupby(level=0)
    .apply(lambda x: x.rr.to_numpy())
    .to_numpy())

# mean
rr_mean = np.mean(L, axis=0)
# rr_mean_dict[animal] = rr_mean
ax.plot(rr_mean)#, label = i, c = colors[i])
sns.despine()
ax.set_xlabel('Trials')
ax.set_ylabel('Reward rate')
plt.tight_layout()

In [ ]:
# bias analysis - this needs to be changed, a la last group meeting (normalize differently)
# set_cwd('/home/rishika/sim/')
# from supplementaryFunctions import *
%matplotlib qt
trialsinsess = 100
# new_unstr_sess_dict = loader('new_unstructured')
# sessdf = new_unstr_sess_dict['/home/rishika/sim/NAS_rishika/4portProb/Box2']
fig = plt.figure(figsize = (8,7))
ax = plt.subplot(111)
filtered = sessdf[~(sessdf.duplicated(subset = ['animal', 'session', 'trialstart', 'trialend', 'eptime'], keep = False))].groupby('session').filter(lambda x: x.reward.size >= trialsinsess)
# take only those sessions
filtered = filtered.groupby('session').head(trialsinsess)

# Precompute choice probabilities
session_sizes = filtered.groupby('session').size()
choice_data = []
for i in range(1, 5):
    port_data = filtered[filtered['port'] == i].groupby(['session', 'rewprob']).size().reset_index(name='count')
    port_data['choice_prob'] = port_data['count'] / port_data['session'].map(session_sizes)
    port_data['port'] = i
    choice_data.append(port_data)
choice_df = pd.concat(choice_data, ignore_index=True)

# Plot
for i in range(1, 5):
    port_df = choice_df[choice_df['port'] == i]
    sns.scatterplot(data=port_df, x='rewprob', y='choice_prob', alpha=0.1, ax=ax)
    mean_df = port_df.groupby('rewprob')['choice_prob'].mean().reset_index()
    sns.lineplot(data=mean_df, x='rewprob', y='choice_prob', ax=ax, linewidth=2, label=f'Port {i}')

ax.set_ylabel('choice probability')
ax.set_xlabel('reward percent')
sns.despine()
ax.legend()
plt.suptitle(f'Choice Probability vs. Reward Percent, per arm, animal = {animal}')
plt.tight_layout()
plt.show()

In [ ]:
# reward collection latency
g = sns.relplot(data=sessdf, x=sessdf['trialend']-sessdf['trialstart'], y='trial', hue='reward')
axes = g.axes.flatten()

# find the last trial number of a session
last = sessdf.groupby('session')['trial'].max()
for ax in axes:
    for l in last:
        ax.axhline(l, linewidth = 0.7, color='grey')
        ax.set_xlabel('Latency (ms)')
        ax.set_xlim(0,5600)
        ax.set_ylim(bottom=-5)
sns.set_context('notebook')
plt.suptitle('Latency', y=1.05)
# plt.savefig('plots/25082023/dms_D03_latency.png', dpi = 300)

Text(0.5, 1.05, 'Latency')

In [ ]:
# number of trials in each session
g = sns.relplot(data=sessdf['session'].value_counts(), kind='scatter', color='k')
g.set_xlabels('session')
g.set_ylabels('# of trials')
# for ax in g.axes.flat:
#     for i in sessdf['session']:
#         if i%3==0:
#             ax.axvline(x=i, color='xkcd:light grey', linestyle='-.')
plt.axhline(np.mean(sessdf['session'].value_counts()), c='r')
# plt.savefig('plots/25082023/dms_D03_numtrials.png', dpi = 300)

C:\Users\dlab\miniforge3\envs\rishika\lib\site-packages\seaborn\axisgrid.py:118: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)


## old plots that may not mean anything

In [ ]:
###### plot cumulative regret 
g = sns.relplot(data = sessdf, x = 'trial', y = np.cumsum(sessdf.regret), color = 'k', kind = 'line')
# find the last trial number of a session
last = sessdf.groupby('session')['trial'].max()
for ax in g.axes.flat:
    for l in last:
        ax.axvline(l, linewidth = 1, color='grey')
ax.set_ylabel('cumulative regret')

Text(-21.30555555555555, 0.5, 'cumulative regret')

In [ ]:
# what is distance moved given last trial was not rewarded?
dist = []
dist_unrew = []
dist_rew = []
for i in range(len(sessdf)):
    if i+1<len(sessdf):
        if sessdf['reward'][i]==0:
            dist_unrew.append(sessdf['port'][i]-sessdf['port'][i+1])
        else:
            dist_rew.append(sessdf['port'][i]-sessdf['port'][i+1])
plt.plot(dist_unrew, 'k')
sns.despine()
# plt.plot(dist_rew, 'r')

# whismur special

In [ ]:
sessdf['blox_status'] = ''
sessdf.loc[(sessdf.datetime >= pd.to_datetime('20260112')) & (sessdf.datetime < pd.to_datetime('20260116')), 'blox_status'] = '2_blocked'
sessdf.loc[(sessdf.datetime >= pd.to_datetime('20260116')) & (sessdf.datetime < pd.to_datetime('20260129')), 'blox_status'] = 'unblocked'
sessdf.loc[(sessdf.datetime >= pd.to_datetime('20260130')) & (sessdf.datetime < pd.to_datetime('20260205')), 'blox_status'] = '23_blocked'
sessdf.loc[(sessdf.datetime >= pd.to_datetime('20260206')), 'blox_status'] = '14_blocked'
sessdf['transition_val'] = sessdf['port'].astype(str).apply(lambda x: x.split('.')[0]) + sessdf['choice_t1'].astype(str).apply(lambda x: x.split('.')[0])

In [ ]:
# 2_blocked switch rate - 1 to 2 vs 1 to 4
from utils.plotSettings import *
parula = get_parula_cmap()
plt.figure()
ax = plt.subplot(221)
empty_cell_val = np.nan
normalizing = 'all'
# empty_cell_val = 0

two_blocked_mask = (sessdf.blox_status == '2_blocked') & ~(sessdf.transition_val.isin(['1nan', '2nan', '3nan', '4nan']))
# add nan row with port 2 in the matrix
two_blocked_tm = pd.crosstab(sessdf[two_blocked_mask].port, sessdf[two_blocked_mask].choice_t1, normalize = normalizing).to_numpy()
two_blocked_tm = np.concatenate([two_blocked_tm[:, :-2], np.ones((3, 1))*empty_cell_val, two_blocked_tm[:, 1:]], axis = 1)
two_blocked_tm = np.concatenate([two_blocked_tm[:-2, :], np.ones((1,4))*empty_cell_val, two_blocked_tm[1:, :]], axis = 0)
sns.heatmap(two_blocked_tm, cmap = parula, vmax = 0.15, vmin = 0, ax = ax, square = True, xticklabels = np.arange(1,5), yticklabels = np.arange(1,5))
ax.set_title('2 blocked')

ax = plt.subplot(222)
unblocked_mask = (sessdf.blox_status == 'unblocked') & ~(sessdf.transition_val.isin(['1nan', '2nan', '3nan', '4nan']))
unblocked_tm = pd.crosstab(sessdf[unblocked_mask].port, sessdf[unblocked_mask].choice_t1, normalize = normalizing)
sns.heatmap(unblocked_tm, cmap = parula, vmax = 0.15, vmin = 0, ax = ax, square = True, xticklabels = np.arange(1,5), yticklabels = np.arange(1,5))
ax.set_title('unblocked')

ax = plt.subplot(223)
centre_blocked_mask = (sessdf.blox_status == '23_blocked') & ~(sessdf.transition_val.isin(['1nan', '2nan', '3nan', '4nan']))
centre_blocked_tm = pd.crosstab(sessdf[centre_blocked_mask].port, sessdf[centre_blocked_mask].choice_t1, normalize = normalizing)#.to_numpy()
centre_blocked_tm[2] = empty_cell_val
centre_blocked_tm[3] = empty_cell_val
centre_blocked_tm.loc[2] = empty_cell_val
centre_blocked_tm.loc[3] = empty_cell_val
centre_blocked_tm = centre_blocked_tm.sort_index()[[1,2,3,4]]
sns.heatmap(centre_blocked_tm, cmap = parula, vmax = 0.15, vmin = 0, ax = ax, square = True, xticklabels = np.arange(1,5), yticklabels = np.arange(1,5))
ax.set_title('2,3 blocked')

ax = plt.subplot(224)
edge_blocked_mask = (sessdf.blox_status == '14_blocked') & ~(sessdf.transition_val.isin(['1nan', '2nan', '3nan', '4nan']))
edge_blocked_tm = pd.crosstab(sessdf[edge_blocked_mask].port, sessdf[edge_blocked_mask].choice_t1, normalize = normalizing)
edge_blocked_tm[1] = empty_cell_val
edge_blocked_tm[4] = empty_cell_val
edge_blocked_tm.loc[1] = empty_cell_val
edge_blocked_tm.loc[4] = empty_cell_val
edge_blocked_tm = edge_blocked_tm.sort_index()[[1,2,3,4]]
sns.heatmap(edge_blocked_tm, cmap = parula, vmax = 0.15, vmin = 0, ax = ax, square = True, xticklabels = np.arange(1,5), yticklabels = np.arange(1,5))
ax.set_title('1,4 blocked')
# ax.patch.set_facecolor('white')

plt.tight_layout()

In [ ]:

plt.bar(np.array([0, 2, 3, 1])-0.4, [unblocked_tm.to_numpy()[3,2]+unblocked_tm.to_numpy()[2,3],
                       unblocked_tm.to_numpy()[0,2]+unblocked_tm.to_numpy()[2,0],
                       unblocked_tm.to_numpy()[0,3]+unblocked_tm.to_numpy()[3,0],
                       unblocked_tm.to_numpy()[1,2]+unblocked_tm.to_numpy()[2,1]], label = 'unblocked', width = 0.4)
plt.bar([0, 2, 3.4], [two_blocked_tm[2,3]+two_blocked_tm[3,2], two_blocked_tm[0, 2]+two_blocked_tm[2,0], two_blocked_tm[0,3]+two_blocked_tm[3, 0]], label = 'two blocked', width = 0.4)
plt.bar([1], edge_blocked_tm.to_numpy()[1, 2]+edge_blocked_tm.to_numpy()[2,1], label = 'edge blocked', width = 0.4)
plt.bar([3], centre_blocked_tm.to_numpy()[0, 3]+centre_blocked_tm.to_numpy()[3,0], label = 'centre blocked', width = 0.4)
plt.ylabel('Switch probability')
plt.xticks(np.array([0-0.2,1-.2,2-.2,3]), ['34, 43', '23, 32', '13, 31', '14, 41'])
plt.legend()
plt.tight_layout()


# Movement time

In [13]:
import pandas as pd
movdf = pd.read_excel(r"C:\Users\dlab\OneDrive - Indian Institute of Science\Data\movement_data.xlsx")
import matplotlib.pyplot as plt
import seaborn as sns
from utils.plotSettings import *
%matplotlib qt

phases = ['late', 'post']
labels = ['Pre-lesion', 'Post-lesion']
colors = [['xkcd:grey', 'xkcd:periwinkle'],
        ['xkcd:grey', 'xkcd:teal']]
fig, axes = plt.subplots(2, 1, figsize = (3.5, 7), sharey = True)

for ind, (ax, group) in enumerate(zip(axes, ['DLS', 'DMS'])):
    sub = movdf[movdf.phase.isin(phases) & (movdf.lesion_group == group)]
    # animal x phase table of median movement times
    y = sub.pivot_table(index = 'animal', columns = 'phase', values = 'median_mt_ms', aggfunc = 'mean')[phases]

    # group mean bars
    ax.bar(x = [0, 1], height = y.mean(), color = colors[ind], width = 0.8, zorder = 1, alpha = 0.8)

    # per-animal points + lines from pre to post
    for animal, row in y.iterrows():
        ax.plot([0, 1], row.values, color = 'xkcd:slate grey', linewidth = 0.7, alpha = 0.5,
                marker = 'o', markersize = 4, zorder = 2)

    # formatting
    ax.set_title(group, fontsize = 'x-large', color = 'grey')
    ax.set_xticks([0, 1], labels, fontsize = 'large', rotation = 0)
    ax.tick_params(axis = 'y', colors = 'grey')
    ax.set_ylabel('Movement time (s)', fontsize = 'x-large')
    ax.set_xlim(-0.6, 1.6)
    ax.set_ylim(0, 3670)
    ax.set_yticklabels(labels=[0, 1, 2, 3])

axes[1].set_xlabel('Condition', fontsize = 'x-large')
plt.tight_layout()

C:\Users\dlab\AppData\Local\Temp\ipykernel_11232\4079497027.py:34: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(labels=[0, 1, 2, 3])
C:\Users\dlab\AppData\Local\Temp\ipykernel_11232\4079497027.py:34: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(labels=[0, 1, 2, 3])
